# 04 - Fine-tune DistilBERT (GPU Required!)
**Author:** Sanjeev Kumar | IIT Bombay

Fine-tune DistilBERT on complaint product classification.
**Runtime: T4 GPU recommended!**

In [ ]:
import pandas as pd, numpy as np, re, os, sys, torch, warnings
import joblib
warnings.filterwarnings('ignore')
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, classification_report
print(f"GPU: {torch.cuda.is_available()}")
if torch.cuda.is_available(): print(torch.cuda.get_device_name(0))

In [ ]:
# Load data
url = "https://files.consumerfinance.gov/ccdb/complaints.csv.zip"
PRODUCT_MAP = {
    'Credit reporting or other personal consumer reports': 'Credit Reporting',
    'Credit reporting, credit repair services, or other personal consumer reports': 'Credit Reporting',
    'Credit reporting': 'Credit Reporting', 'Debt collection': 'Debt Collection',
    'Mortgage': 'Mortgage', 'Checking or savings account': 'Bank Account',
    'Bank account or service': 'Bank Account', 'Credit card': 'Credit Card',
    'Credit card or prepaid card': 'Credit Card', 'Prepaid card': 'Credit Card',
    'Student loan': 'Loans', 'Vehicle loan or lease': 'Loans', 'Consumer Loan': 'Loans',
    'Payday loan, title loan, personal loan, or advance loan': 'Loans',
    'Payday loan, title loan, or personal loan': 'Loans', 'Payday loan': 'Loans',
    'Money transfer, virtual currency, or money service': 'Money Transfer',
    'Money transfers': 'Money Transfer', 'Debt or credit management': 'Debt Collection',
    'Other financial service': 'Other',
}
def clean_text(t):
    if pd.isna(t): return ""
    t = str(t).lower()
    t = re.sub(r'x{2,}', 'XXXX', t)
    return re.sub(r'\s+', ' ', t).strip()

df = pd.read_csv(url, compression='zip', low_memory=False, nrows=200000)
df_text = df[df['Consumer complaint narrative'].notna()].copy()
df_text['product_clean'] = df_text['Product'].map(PRODUCT_MAP).fillna('Other')
df_text['narrative_clean'] = df_text['Consumer complaint narrative'].apply(clean_text)
df_text['Date received'] = pd.to_datetime(df_text['Date received'])
df_sorted = df_text.sort_values('Date received').reset_index(drop=True)
n = len(df_sorted)
train_df = df_sorted.iloc[:int(n*0.70)]
val_df   = df_sorted.iloc[int(n*0.70):int(n*0.85)]
test_df  = df_sorted.iloc[int(n*0.85):]
X_train = train_df['narrative_clean'].values
X_val   = val_df['narrative_clean'].values
X_test  = test_df['narrative_clean'].values
y_train_prod = train_df['product_clean'].values
y_val_prod   = val_df['product_clean'].values
y_test_prod  = test_df['product_clean'].values
le = LabelEncoder()
le.fit(y_train_prod)
print(f"Classes: {list(le.classes_)}")
print(f"Train:{len(train_df):,} Val:{len(val_df):,} Test:{len(test_df):,}")

In [ ]:
# Tokenize
MODEL_NAME = "distilbert-base-uncased"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=256)

train_ds = Dataset.from_dict({'text': list(X_train), 'label': le.transform(y_train_prod).tolist()})
val_ds   = Dataset.from_dict({'text': list(X_val),   'label': le.transform(y_val_prod).tolist()})
test_ds  = Dataset.from_dict({'text': list(X_test),  'label': le.transform(y_test_prod).tolist()})

train_ds = train_ds.map(tokenize, batched=True, batch_size=256)
val_ds   = val_ds.map(tokenize, batched=True, batch_size=256)
test_ds  = test_ds.map(tokenize, batched=True, batch_size=256)

for ds in [train_ds, val_ds, test_ds]:
    ds.set_format('torch', columns=['input_ids','attention_mask','label'])
print("Tokenization complete!")

In [ ]:
# Fix torchvision conflict if needed
import sys
mods = [m for m in sys.modules if 'torchvision' in m or 'datasets' in m]
for m in mods: del sys.modules[m]

# Load model
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(le.classes_), ignore_mismatched_sizes=True)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {'macro_f1': f1_score(labels, preds, average='macro'),
            'weighted_f1': f1_score(labels, preds, average='weighted')}

training_args = TrainingArguments(
    output_dir='./distilbert_complaint',
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    logging_steps=100,
    fp16=torch.cuda.is_available(),
    seed=42,
    report_to='none'
)
trainer = Trainer(
    model=model, args=training_args,
    train_dataset=train_ds, eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    processing_class=tokenizer
)
print("\nStarting fine-tuning (15-25 mins on GPU)...")
trainer.train()

In [ ]:
# Evaluate on test set
print("Evaluating on OOT Test Set...")
test_results = trainer.predict(test_ds)
test_preds   = np.argmax(test_results.predictions, axis=-1)
test_labels  = le.inverse_transform(test_preds)
test_f1_bert = f1_score(y_test_prod, test_labels, average='macro')

print("="*65)
print("FINAL PRODUCT CLASSIFICATION COMPARISON")
print("="*65)
print(f"{'Model':<35} {'Test F1':>8} {'Decision'}")
print("-"*65)
print(f"{'TF-IDF + LR':<35} {'0.7652':>8}  Champion!")
print(f"{'TF-IDF + SVM':<35} {'0.7591':>8}  Strong baseline")
print(f"{'Word2Vec + XGB':<35} {'0.6383':>8}  Underperformed")
print(f"{'Frozen BERT + LR':<35} {'0.6737':>8}  Underperformed")
print(f"{'Fine-tuned DistilBERT':<35} {test_f1_bert:>8.4f}  Challenger")

lift = test_f1_bert - 0.7652
print(f"\nBERT lift over TF-IDF: {lift:+.4f}")
if lift >= 0.02:
    print("BERT wins! Use as final model!")
else:
    print("Marginal lift. TF-IDF stays as champion!")
    print("BERT = challenger in portfolio")

print("\nClassification Report:")
print(classification_report(y_test_prod, test_labels))

# Save
trainer.save_model('../models/distilbert_finetuned')
print("Model saved!")

## Model Selection Decision

| Model | Test Macro-F1 | Decision |
|-------|--------------|----------|
| TF-IDF + LR | 0.7652 | **Champion** |
| Fine-tuned DistilBERT | ~0.772 | Challenger (+0.007) |

> BERT gave only marginal lift (+0.007). TF-IDF + LR selected as production champion due to speed, interpretability, and governance. This is **senior DS thinking** — not blindly using BERT!

**Next → 05_complaint_triage.ipynb**